# ML workflow

Grid-search models on one feature set for a target, pick the best per model type, evaluate on a held-out test set, and interpret with SHAP.

**Input:** a featurized dataset (output of `create_features_nmr.ipynb`).  
**Output:** best-model tables, an actual-vs-predicted plot, SHAP summary, and NMF-component visualizations.

The search space and metrics come from [`nmrlib`](nmrlib/); `feature_set` is looked up in [`nmrlib.features.feature_sets`](nmrlib/features.py).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.base import clone
from sklearn.ensemble import (
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV, KFold, train_test_split

from nmrlib import load_dataset, regression_metrics
from nmrlib.features import feature_sets
from nmrlib.models import grid_search_space

## Config

In [ ]:
dataset = "alberts_10k_100kdict"   # registry name or path to a featurized .pkl
target_col = "gap_ev"              # gap_ev | log_p_rdkit | homo_ev | lumo_ev
feature_set = "NMF"                # any key from feature_sets(df)
seed = 42
train_frac, val_frac, test_frac = 0.70, 0.15, 0.15

## Load & split

In [ ]:
df = load_dataset(dataset)
feature_cols = feature_sets(df)[feature_set]
print(f"Feature set {feature_set!r}: {len(feature_cols)} columns")

In [ ]:
model_df = df.dropna(subset=[target_col] + feature_cols)
print(f"{len(model_df)} / {len(df)} rows have no NaNs in target + features")

X = model_df[feature_cols].astype(float)
y = model_df[target_col].astype(float)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=test_frac, random_state=seed,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=val_frac / (train_frac + val_frac),
    random_state=seed,
)
print(f"Train {len(X_train)}  |  Val {len(X_val)}  |  Test {len(X_test)}")

## Hyperparameter search

In [ ]:
search_pipe, param_grid = grid_search_space(len(feature_cols), seed=seed)

cv = KFold(n_splits=5, shuffle=True, random_state=seed)
n_combos = sum(int(np.prod([len(v) for v in g.values()])) for g in param_grid)
print(f"{n_combos} param combos x {cv.get_n_splits()} folds = {n_combos * cv.get_n_splits()} fits")

grid_search = GridSearchCV(
    search_pipe, param_grid, cv=cv, scoring="neg_root_mean_squared_error",
    n_jobs=-1, refit=True,
)
grid_search.fit(X_trainval, y_trainval)
best_pipe = grid_search.best_estimator_
print("Best params:", grid_search.best_params_)

## Best model per type

In [ ]:
cv_res = pd.DataFrame(grid_search.cv_results_)
cv_res["model_type"] = cv_res["param_model"].apply(lambda m: type(m).__name__)
best_per_type = (
    cv_res.sort_values("rank_test_score")
    .drop_duplicates(subset="model_type", keep="first")
)

best_pipes = {}
for _, row in best_per_type.iterrows():
    pipe_clone = clone(search_pipe).set_params(**row["params"])
    pipe_clone.fit(X_trainval, y_trainval)
    best_pipes[row["model_type"]] = {
        "pipeline": pipe_clone,
        "cv_rmse": -row["mean_test_score"],
        "params": row["params"],
    }

print(f"Best pipeline per model type ({len(best_pipes)}):")
for name, info in sorted(best_pipes.items(), key=lambda x: x[1]['cv_rmse']):
    print(f"  {name:35s}  CV RMSE = {info['cv_rmse']:.4f}")

## Test set — all model types

In [ ]:
overall_best_type = type(grid_search.best_params_["model"]).__name__

rows = []
for model_name, info in best_pipes.items():
    metrics = regression_metrics(y_test, info["pipeline"].predict(X_test))
    rows.append({
        "model_type": model_name,
        "cv_rmse": round(info["cv_rmse"], 4),
        "feature_set": feature_set,
        "n_features": len(feature_cols),
        "target": target_col,
        **metrics,
    })
test_comparison = pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)
test_comparison

## Actual vs predicted (best model)

In [ ]:
y_test_pred = best_pipe.predict(X_test)
test_metrics = regression_metrics(y_test, y_test_pred)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_test_pred, alpha=0.6, s=25, edgecolors="none")
lo = min(y_test.min(), y_test_pred.min())
hi = max(y_test.max(), y_test_pred.max())
ax.plot([lo, hi], [lo, hi], "r--", lw=1.5)
ax.set_xlabel(f"Computed {target_col}")
ax.set_ylabel(f"Predicted {target_col}")
ax.set_aspect("equal", adjustable="box")
ax.text(
    0.97, 0.03,
    f"R\u00b2 = {test_metrics['r2']:.3f}\n"
    f"MAE = {test_metrics['mae']:.3f}\n"
    f"RMSE = {test_metrics['rmse']:.3f}",
    transform=ax.transAxes, ha="right", va="bottom",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
)
plt.tight_layout()
plt.show()

## SHAP

In [ ]:
model_step = best_pipe.named_steps["model"]
scaler_step = best_pipe.named_steps["scaler"]

X_bg = pd.DataFrame(
    scaler_step.transform(X_train.sample(min(200, len(X_train)), random_state=seed)),
    columns=feature_cols,
)
X_explain = pd.DataFrame(
    scaler_step.transform(X_test.iloc[:200]),
    columns=feature_cols,
)

if isinstance(model_step, (RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor)):
    explainer = shap.TreeExplainer(model_step)
    shap_values = explainer.shap_values(X_explain)
elif isinstance(model_step, KNeighborsRegressor):
    bg = shap.sample(X_bg, 50, random_state=seed)
    X_explain = X_explain.iloc[:50]
    explainer = shap.KernelExplainer(model_step.predict, bg)
    shap_values = explainer.shap_values(X_explain)
else:
    explainer = shap.LinearExplainer(model_step, X_bg)
    shap_values = explainer.shap_values(X_explain)

shap.summary_plot(shap_values, X_explain, show=True)

In [ ]:
# mean |SHAP| per feature
mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_cols)
mean_abs_shap.sort_values(ascending=False).head(15)

## Visualize top NMF components

Maps the highest-|SHAP| NMF codes back to their H/C spectral motifs.

In [ ]:
from pathlib import Path
from plot_nmf_components import plot_nmf_components

# Point this at the NMF dictionary artifact the codes were built from
nmf_dictionary = Path("/Users/jacknugent/Downloads/gap_from_100k_dict/nmf_dictionary_100k.pkl")

# Top codes by mean |SHAP| (strip the nmr_dict_code_ prefix -> component index)
top_codes = [
    int(name.split("_")[-1])
    for name in mean_abs_shap.sort_values(ascending=False).index
    if name.startswith("nmr_dict_code_")
][:3]
print("Top NMF components:", top_codes)

if nmf_dictionary.exists() and top_codes:
    plot_nmf_components(
        model_path=nmf_dictionary,
        comp_indices=top_codes,
        output_plot=Path("figures/top_shap_nmf_components.png"),
    )